# 01. Binance Live Crypto Data Exploration
This notebook demonstrates how to load credentials from `.env`, query public/private Binance endpoints using `BinanceClient`, fetch top 24h gainers/volume pairs, and format market data for exploratory analysis.

In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import plotly.express as px
from backend.binance_client import binance_client
from backend.config import API_KEY

print(f"Binance API Key Configured: {'Yes' if API_KEY else 'No'}")

## 1. Fetch Top 24h USDT Volume Leaders

In [ ]:
usdt_tickers = binance_client.get_usdt_tickers()
df_tickers = pd.DataFrame(usdt_tickers)

# Clean numeric fields
numeric_cols = ['lastPrice', 'priceChangePercent', 'quoteVolume', 'volume', 'highPrice', 'lowPrice']
for col in numeric_cols:
    df_tickers[col] = df_tickers[col].astype(float)

# Sort by 24h volume
df_top_vol = df_tickers.sort_values(by='quoteVolume', ascending=False).head(20)
df_top_vol[['symbol', 'lastPrice', 'priceChangePercent', 'quoteVolume']].head(10)

## 2. Visualize 24h Volume Distribution & Top Gainers

In [ ]:
fig = px.bar(
    df_top_vol.head(15),
    x='symbol',
    y='quoteVolume',
    color='priceChangePercent',
    color_continuous_scale='RdYlGn',
    title='Top 15 Binance USDT Pairs by 24h Quote Volume ($)',
    labels={'quoteVolume': '24h Volume ($)', 'priceChangePercent': '24h Change (%)'}
)
fig.show()

## 3. Fetch Historical Candlesticks (BTC/USDT)

In [ ]:
from backend.indicators import enrich_klines_dataframe

raw_klines = binance_client.get_klines('BTCUSDT', interval='1h', limit=100)
df_btc = enrich_klines_dataframe(raw_klines)
df_btc[['open_time', 'open', 'high', 'low', 'close', 'volume', 'rsi', 'macd']].tail(10)